# G1：ROS 2 话题与服务 —— 分布式通信
# G1: ROS 2 Topics & Services — Distributed Communication

## 一、实验原理 | Principle

### 1.1 什么是 ROS 2？
**ROS 2**（Robot Operating System）：机器人/自动驾驶的事实标准中间件，
提供进程间通信、硬件抽象、工具链。Autoware（开源自动驾驶栈）就基于 ROS 2。

### 1.2 三种通信模式
- **话题（Topic）**：异步发布/订阅。如 `/vehicle/speed`——车辆每 0.05s 广播速度，
  谁想听就订阅，不用应答。适合高频状态流。
- **服务（Service）**：同步请求/响应。如 `/get_map`——请求一次，应答一次。适合低频查询。
- **动作（Action）**：长时间任务。如导航——可取消、可反馈进度。

### 1.3 在 IEV CloudLab 中的定位
> **重要**：本平台中 ROS 2 **不做车辆闭环控制**！
> A1/B3 的控制器直调仿真器 API（延迟 <1ms），
> 走 ROS 2 话题会有 5-10ms 延迟，不适合实时控制。
>
> ROS 2 用于：**多车状态广播**、**参数服务**、**数据记录**（如 `foxglove_bridge` 可视化）。


### 1.4 QoS 服务质量 | Quality of Service
ROS 2 话题的 QoS 策略决定消息传递的可靠性：

| 策略 | 选项 | 说明 |
|---|---|---|
| 可靠性 Reliability | Reliable / Best-Effort | Reliable 重传丢包；Best-Effot 不保证 |
| 历史 History | Keep-Last(N) / Keep-All | 新订阅者能收到几条旧消息 |
| 持久性 Durability | Volatile / Transient-Local | 话题是否为"有状态" |

**教学要点**：默认 QoS 下话题是 Best-Effort——消息可能丢失。这再次说明话题不适合闭环控制（丢一帧速度，控制器就用过期数据）。服务（Service）则是有应答的，可靠性由协议保证。

### 1.5 消息流分析 | Message Flow
```
[Publisher] --(DDS/RTPS)--> [DDS Domain] --(DDS/RTPS)--> [Subscriber]
     |                            |                            |
  rclpy API                  中间件层                    rclpy 回调
```
- 应用层只看到 `publish()` / 回调函数；
- 中间件（Fast DDS）负责发现、序列化、传输；
- 本机回环延迟约 0.1-1ms，但加上 Python 回调调度、GIL 竞争，端到端常达 5-10ms。

## 二、实验目的 | Objectives
1. 理解话题 vs 服务 vs 动作的适用场景
2. 掌握 ROS 2 在 IEV CloudLab 架构中的定位（什么用、什么不用）
3. 会用 `ros2 topic` 命令行工具

## 三、学生任务 | Your Tasks
1. **运行**：MiniROS 模拟，理解发布/订阅模式
2. **终端实操**（在 JupyterLab 点 Terminal）：
   ```
   source /opt/ros/jazzy/setup.bash
   ros2 run demo_nodes_cpp talker  # 新开一个终端
   ros2 topic echo /chatter          # 观察消息流
   ```
3. **思考题**：为什么闭环控制不用 ROS 2 话题？（提示：延迟、抖动、实时性）

## 四、验证内容 | Verification
- [ ] 理解三种通信模式的区别
- [ ] MiniROS 模拟通过（收到 3 条消息）
- [ ] （可选）终端跑通 talker/listener

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `ros2: command not found` | 未 source 环境 | 先 `source /opt/ros/jazzy/setup.bash` |
| `No such file or directory` | demo 节点未安装 | `apt install ros-jazzy-demo-nodes-cpp` |


In [ ]:
import os
print('ROS 2 Jazzy 可用:', os.path.exists('/opt/ros/jazzy'))
# ROS 2 环境（notebook 中演示概念，完整节点需终端运行）
print('\\nROS 2 核心概念:')
print('  - Topic: 异步发布/订阅（如 /vehicle/speed）')
print('  - Service: 同步请求/响应（如 /get_map）')
print('  - Action: 长时间任务（如导航）')


## 1. 话题通信模式（概念演示）

In [ ]:
# 用 Python 模拟 ROS 2 话题的发布/订阅模式
from collections import defaultdict
import time

class MiniROS:
    """极简 ROS 2 话题模拟"""
    def __init__(self):
        self.subs = defaultdict(list)
    def publish(self, topic, msg):
        for cb in self.subs[topic]:
            cb(msg)
    def subscribe(self, topic, callback):
        self.subs[topic].append(callback)

ros = MiniROS()
received = []
ros.subscribe('/vehicle/speed', lambda m: received.append(m))
# 模拟车辆速度发布
for v in [15.0, 14.5, 14.0]:
    ros.publish('/vehicle/speed', {'v': v, 't': time.time()})
print(f'收到 {len(received)} 条速度消息: {[m["v"] for m in received]}')
assert len(received) == 3
print('✅ 话题通信验证通过！')


## 2. 在 IEV CloudLab 中的定位
- **不用 ROS 做闭环**：A1/B3 的控制器直接调仿真器 API，延迟更低
- **用 ROS 做什么**：多车协同的状态广播、参数服务、数据记录
- **foxglove_bridge**：WebSocket 桥接，可视化（见设计文档 §4）

## 五、结果分析 | 5. Results Analysis

### 5.1 结果概述 | Overview
| 检查项 | 结果 |
|---|---|
| MiniROS 发布/订阅模拟 | ✅ 收到 3 条速度消息 |
| ROS 2 Jazzy 环境 | ✅ `/opt/ros/jazzy` 存在 |
| 三种通信模式理解 | ✅ 话题/服务/动作区分清楚 |

### 5.2 话题通信特性分析 | Topic Characteristics
MiniROS 模拟验证了话题的核心语义：
- **异步**：发布者只管发，不等待订阅者确认；
- **多对多**：一个话题可有多个发布者和多个订阅者；
- **消息丢失可接受**：本例 3 条全收到，但 QoS 配置为 Best-Effort 时丢包是正常现象——这正是话题**不适合闭环控制**的原因之一。

### 5.3 延迟定性分析 | Latency (Qualitative)
| 路径 | 典型延迟 | 是否适合 20 Hz 闭环 |
|---|---|---|
| 控制器直调仿真器 API | <1 ms | ✅ |
| ROS 2 话题（本机回环） | 5-10 ms | ❌（占周期 10-20%） |
| ROS 2 话题（跨机器） | 10-50 ms + 抖动 | ❌ |

**结论**：A1/B3 的控制器直调 API 是架构上的必然选择；ROS 2 在本平台的定位是**状态广播、参数服务、数据记录**（如 `foxglove_bridge` 可视化），详见 §1.3。

### 5.4 工程启示 | Takeaway
Autoware 等真实自动驾驶栈确实用 ROS 2 做模块间通信，但它们的**底层控制**（如线控执行）同样绕过 ROS 2 走实时总线（CAN）。分层思想一致：实时层用专用通道，非实时层用 ROS 2。


## 🏆 挑战任务
1. 在终端启动 `ros2 run demo_nodes_cpp talker`，用 `ros2 topic echo` 观察。
2. 设计一个 `/platoon/status` 话题的消息格式。

---

## 下一个实验 | Next →

**[G2_ros_services](./G2_ros_services.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
